In [ ]:

import os
import json
import joblib
import pandas as pd
import numpy as np
import torch
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold


In [ ]:
"""
Baseline RF para detectar y corregir etiquetas infectadas (equivalente al RF del
artículo): validación cruzada de 4 folds SEPARADA POR PARCELA, predicciones
fuera de fold, regla del 75 % por parcela y métricas de recall y precisión por
parcela (las mismas que usa el CAE).

NOVEDAD: además de la predicción, se guardan las probabilidades OOF medias por
parcela y clase (`prob_media`) en resultado_parcelas.npz. Con eso se puede aplicar
Otsu después, sin reentrenar (error de la parcela bajo la clase c = 1 - prob_media[:, c]).
"""

dir_base_datasets = "/home/angelo/MyStore/Proyecto Investigador/2017/entrenamiento train fiables test/dataset_ruidoso_farmsar_test_v1/"
dir_base_resultados = "/home/angelo/MyStore/Proyecto Investigador/2017/entrenamiento train fiables test/resultados_RF_backscatter_v1/"
os.makedirs(dir_base_resultados, exist_ok=True)

noise_levels = [1, 5, 10, 15, 20, 25, 30]
n_repetitions = 10
N_FOLDS = 4
UMBRAL = 0.75

# class_weight="balanced_subsample" es decisión tuya; el artículo no la especifica.
RF_PARAMS = dict(
    n_estimators=500,
    max_depth=None,
    min_samples_split=10,
    min_samples_leaf=4,
    class_weight="balanced_subsample",
    max_features="sqrt",
    n_jobs=-1,
)


def diagnostico(orig, inf, final, relabel):
    infectada = inf != orig
    return {
        "infectadas": int(infectada.sum()),
        "relabels": int(relabel.sum()),
        "acierto": int((relabel & infectada & (final == orig)).sum()),       # devuelta a su clase original
        "mal_destino": int((relabel & infectada & (final != orig)).sum()),   # detectada, pero a otra clase
        "no_detectada": int((infectada & ~relabel).sum()),                   # infectada sin relabel
        "falso_positivo": int((relabel & ~infectada).sum()),                 # parcela limpia relabelada
    }


def relabel_por_parcela(candidatas, y_inf, pid, umbral=UMBRAL):
    """
    Una parcela se relabela si más del 75 % de sus píxeles comparten la misma
    clase candidata y esa clase difiere de su etiqueta (infectada).
    Devuelve ids de parcela (ordenados), etiqueta infectada, etiqueta final, máscara relabel.
    """
    upid, first, inv = np.unique(pid, return_index=True, return_inverse=True)
    etiqueta = y_inf[first]
    clases, cinv = np.unique(candidatas, return_inverse=True)
    M = np.zeros((len(upid), len(clases)))
    np.add.at(M, (inv, cinv), 1)
    frac = M / M.sum(1, keepdims=True)
    nueva = clases[frac.argmax(1)]
    relabel = (frac.max(1) > umbral) & (nueva != etiqueta)
    return upid, etiqueta, np.where(relabel, nueva, etiqueta), relabel


def cargar_replica(ruta_pt):
    data = torch.load(ruta_pt, map_location="cpu", weights_only=False)
    datos, meta = data["train"], data["meta"]
    X, y_inf, pid = [], [], []
    for c in sorted(datos):
        Xc = torch.as_tensor(datos[c]["X"], dtype=torch.float32)
        Xc = Xc.reshape(len(Xc), -1).numpy()                 # (N, 2, 61) -> (N, 122)
        X.append(Xc)
        y_inf.append(np.full(len(Xc), c))
        pid.append(np.asarray(datos[c]["parcelas"]))
    return np.concatenate(X), np.concatenate(y_inf), np.concatenate(pid), meta


def predecir_fuera_de_fold_y_guardar(X, y, pid, seed, dir_modelos_fold, guardar_modelos=True):
    """
    Devuelve:
        oof    : clase predicha por píxel (fuera de fold)
        probs  : probabilidades OOF por píxel, columnas alineadas con `clases`
        clases : np.unique(y) (orden de las columnas de probs)
    """
    clases = np.unique(y)
    oof = np.empty(len(y), dtype=y.dtype)
    probs = np.zeros((len(y), len(clases)), dtype=np.float32)

    if guardar_modelos:
        os.makedirs(dir_modelos_fold, exist_ok=True)
    cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)

    for fold_idx, (tr, te) in enumerate(cv.split(X, y, groups=pid)):
        rf = RandomForestClassifier(**RF_PARAMS, random_state=seed)
        rf.fit(X[tr], y[tr])

        p = rf.predict_proba(X[te])
        cols = np.searchsorted(clases, rf.classes_)   # por si a un fold le falta una clase
        probs[np.ix_(te, cols)] = p
        oof[te] = rf.classes_[p.argmax(1)]            # equivale a rf.predict(X[te])

        if guardar_modelos:
            joblib.dump(rf, os.path.join(dir_modelos_fold, f"rf_fold_{fold_idx}.pkl"))

    return oof, probs, clases


def prob_media_por_parcela(probs, pid):
    """(n_pix, n_clases) -> (n_parcelas, n_clases), en el mismo orden que np.unique(pid)."""
    upid, inv = np.unique(pid, return_inverse=True)
    cnt = np.bincount(inv, minlength=len(upid))
    return np.stack([np.bincount(inv, weights=probs[:, j], minlength=len(upid)) / cnt
                     for j in range(probs.shape[1])], axis=1).astype(np.float32)


def evaluar_rf_replica(pct, rep, guardar_modelos=True):
    ruta_pt = os.path.join(dir_base_datasets, f"dataset_noise_{pct:02d}pct_rep_{rep:02d}.pt")
    if not os.path.exists(ruta_pt):
        print(f"No existe: {ruta_pt}")
        return None

    dir_rep = os.path.join(dir_base_resultados, f"ruido_{pct:02d}pct", f"rep_{rep:02d}")
    os.makedirs(dir_rep, exist_ok=True)
    f_json = os.path.join(dir_rep, "metricas_rf.json")
    f_npz = os.path.join(dir_rep, "resultado_parcelas.npz")

    # Reanudar solo si el npz ya incluye las probabilidades; si no, se recalcula
    old_relabel = None
    if os.path.exists(f_json) and os.path.exists(f_npz):
        z = np.load(f_npz, allow_pickle=True)
        if "prob_media" in z.files:
            with open(f_json) as fh:
                return json.load(fh)
        old_relabel = z["relabel"]                  # para verificar reproducibilidad

    seed = 1000 * pct + rep                         # misma semilla que el CAE
    X, y_inf, pid, meta = cargar_replica(ruta_pt)
    oof, probs, clases = predecir_fuera_de_fold_y_guardar(
        X, y_inf, pid, seed, os.path.join(dir_rep, "modelos"), guardar_modelos
    )

    upid, etiq_inf, final, relabel = relabel_por_parcela(oof, y_inf, pid)
    prob_media = prob_media_por_parcela(probs, pid)

    if old_relabel is not None and not np.array_equal(old_relabel, relabel):
        print(f"  AVISO: el relabel nuevo difiere del guardado antes "
              f"({int((old_relabel != relabel).sum())} parcelas)")

    inf_map = dict(zip(meta["ids_fiables"], meta["etiqueta_infectada"]))
    orig_map = dict(zip(meta["ids_fiables"], meta["clase_fiable"]))
    assert all(str(inf_map[p]) == str(e) for p, e in zip(upid, etiq_inf)), \
        "Las etiquetas del dataset no coinciden con meta['etiqueta_infectada']"
    orig = np.array([orig_map[p] for p in upid])

    d = diagnostico(orig, etiq_inf, final, relabel)
    d.update(recall=d["acierto"] / max(d["infectadas"], 1),
             precision=d["acierto"] / max(d["relabels"], 1),
             pct=pct, rep=rep, n_parcelas=int(len(upid)))

    np.savez(f_npz, upid=upid, orig=orig, inf=etiq_inf, final=final, relabel=relabel,
             prob_media=prob_media, clases=clases)
    with open(f_json, "w") as fh:
        json.dump(d, fh)
    return d


def ejecutar(niveles=noise_levels, n_rep=n_repetitions, guardar_modelos=True):
    resultados = []
    for pct in niveles:
        for rep in range(n_rep):
            print(f"\nRF | ruido {pct}% | réplica {rep:02d}")
            d = evaluar_rf_replica(pct, rep, guardar_modelos)
            if d:
                resultados.append(d)
                print(f"  infectadas {d['infectadas']} | relabels {d['relabels']} | "
                      f"recall {d['recall']:.2f} | precisión {d['precision']:.2f}")
    return resultados  # ← retornar

# Prueba primero con una sola réplica para medir el tiempo:
# ejecutar(niveles=[1], n_rep=1)
# Después, todo:
# ejecutar(niveles=noise_levels, n_rep=n_repetitions)


In [ ]:
resultados = ejecutar(niveles=noise_levels, n_rep=n_repetitions)


RF | ruido 1% | réplica 00
  infectadas 12 | relabels 15 | recall 1.00 | precisión 0.80

RF | ruido 1% | réplica 01
  infectadas 12 | relabels 16 | recall 1.00 | precisión 0.75

RF | ruido 1% | réplica 02
  infectadas 12 | relabels 15 | recall 1.00 | precisión 0.80

RF | ruido 1% | réplica 03
  infectadas 6 | relabels 11 | recall 1.00 | precisión 0.55

RF | ruido 1% | réplica 04
  infectadas 10 | relabels 13 | recall 1.00 | precisión 0.77

RF | ruido 1% | réplica 05
  infectadas 12 | relabels 17 | recall 1.00 | precisión 0.71

RF | ruido 1% | réplica 06
  infectadas 10 | relabels 13 | recall 1.00 | precisión 0.77

RF | ruido 1% | réplica 07
  infectadas 8 | relabels 11 | recall 1.00 | precisión 0.73

RF | ruido 1% | réplica 08
  infectadas 12 | relabels 16 | recall 1.00 | precisión 0.75

RF | ruido 1% | réplica 09
  infectadas 10 | relabels 14 | recall 1.00 | precisión 0.71

RF | ruido 5% | réplica 00
  infectadas 56 | relabels 60 | recall 0.98 | precisión 0.92

RF | ruido 5% | réplic

In [ ]:
# ============================================================
# GUARDAR RESUMEN
# ============================================================

df_resultados = pd.DataFrame(resultados)

ruta_resumen = os.path.join(
    dir_base_resultados,
    "resumen_entrenamiento_rf.csv"
)

df_resultados.to_csv(ruta_resumen, index=False)

print(df_resultados.to_string())
print(f"\nResumen guardado en: {ruta_resumen}")

NameError: name 'resultados' is not defined